# Address correction inference

Runs the address corrector from `src/` on Bangla addresses. Reference names come from `data/bangladesh_addresses_data.json`.

1. **Embed** every `label: value` query with BGE-M3 in one batch
2. **Vector search + re-rank** the fields in order:
   1. **District**: search all 64 districts, re-rank with Bangla character n-grams
   2. **Thana**: search only the thanas whose `district` metadata is the corrected district, then re-rank
   3. **Post office**: search only the thanas whose `district` metadata is the corrected district, then re-rank
3. **Compose** the corrected address

Settings come from `config.yaml`.

## Setup

In [1]:
import logging
import sys
from html import escape
from pathlib import Path

from IPython.display import HTML, display

# Make the project root importable, wherever the kernel was started
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import AddressCorrector, Trace, get_settings

# Every pipeline step is also sent to `logging`; keep the notebook output to warnings and up
logging.basicConfig(level=logging.WARNING)

d:\codes\unified_projects\embedding-based-address-corrector\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def table(rows: list[dict], columns: list[str] | None = None) -> HTML:
    """Render a list of dicts as an HTML table."""
    columns = columns or (list(rows[0]) if rows else [])
    fmt = lambda v: f"{v:.4f}" if isinstance(v, float) else escape(str(v))
    head = "".join(f"<th>{escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{fmt(row.get(c, ''))}</td>" for c in columns) + "</tr>" for row in rows)
    return HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>")


def highlight(result: dict) -> HTML:
    """The corrected address with substituted values marked (green: corrected, grey: already correct)."""
    parts = []
    for segment in result["segments"]:
        if "collection" not in segment:
            parts.append(escape(segment["text"]))
        elif segment["changed"]:
            parts.append(f'<s style="color:#c0392b">{escape(segment["original"])}</s> '
                         f'<b style="color:#1e8449">{escape(segment["text"])}</b>')
        else:
            parts.append(f'<b style="color:#7f8c8d">{escape(segment["text"])}</b>')
    return HTML(f'<p style="font-size:1.1em;line-height:1.8">{"".join(parts)}</p>')

## Configuration

In [3]:
settings = get_settings()
display(table([
    {"setting": "model", "value": settings.model_name},
    {"setting": "model_path", "value": settings.model_path},
    {"setting": "device", "value": settings.device or "auto"},
    {"setting": "chroma_path", "value": settings.chroma_path},
    {"setting": "data_path", "value": settings.data_path},
    {"setting": "n_results", "value": settings.n_results},
    {"setting": "padding", "value": settings.padding},
]))

setting,value
model,BAAI/bge-m3
model_path,D:\codes\unified_projects\embedding-based-address-corrector\models\bge-m3
device,auto
chroma_path,D:\codes\unified_projects\embedding-based-address-corrector\chroma_db
data_path,D:\codes\unified_projects\embedding-based-address-corrector\data\bangladesh_addresses_data.json
n_results,30
padding,0


## Load the corrector

Loads the reference locations and the embedding model, then opens ChromaDB. Collections are only re-embedded when the reference names, their metadata or the model changed since the last run. The first load takes a while (the model is about 2 GB, and embedding all post offices on a CPU takes a few minutes).

In [4]:
startup = Trace()
corrector = AddressCorrector.from_settings(settings, startup)

display(table(startup.events, ["elapsed_ms", "level", "step", "message"]))
display(table(corrector.store.stats(), ["collection", "label", "count", "reference_count", "filter_by", "up_to_date", "status"]))

Loading weights: 100%|██████████| 391/391 [00:02<00:00, 179.44it/s]


elapsed_ms,level,step,message
0.1000,info,startup,Loading embedding model 'BAAI/bge-m3'
22706.1300,success,startup,Model loaded from D:\codes\unified_projects\embedding-based-address-corrector\models\bge-m3 on cpu in 22.7 s (dimension 1024)
23046.9200,info,startup,"Loaded reference locations from D:\codes\unified_projects\embedding-based-address-corrector\data\bangladesh_addresses_data.json: 68 district, 915 thana, 10126 post office"
23046.9700,info,startup,Opening ChromaDB at D:\codes\unified_projects\embedding-based-address-corrector\chroma_db
24920.1800,info,ingest,"[District] Skipping, collection is up to date"
25405.6300,info,ingest,"[Thana] Skipping, collection is up to date"
25803.5100,info,ingest,"[Post office] Skipping, collection is up to date"


collection,label,count,reference_count,filter_by,up_to_date,status
district,জেলা,68,68,None,True,up to date
thana,থানা,915,915,district,True,up to date
post_office,ডাকঘর,10126,10126,district,True,up to date


### Thanas per district

Each thana record stores its district as metadata. Thana search only compares the thanas of the corrected district, which is a small fraction of the collection.

In [5]:
from collections import Counter

thana_field = next(f for f in corrector.fields if f.collection == "thana")
per_district = Counter(dict(r.metadata)["district"] for r in thana_field.unique_records)
print(f"{len(thana_field.unique_records)} thana records across {len(per_district)} districts, "
      f"{min(per_district.values())} to {max(per_district.values())} per district "
      f"(mean {len(thana_field.unique_records) / len(per_district):.1f})")
display(table([{"district": d, "thanas": n} for d, n in per_district.most_common(5)]))

915 thana records across 68 districts, 4 to 77 per district (mean 13.5)


district,thanas
ঢাকা,77
চট্টগ্রাম,44
রাজশাহী,31
খুলনা,25
সুনামগঞ্জ,24


## Correct one address

In [12]:
address = "রহমান মঞ্জিল, বাড়ি নং: ৩৪, রোড নং: ৬, সেক্টর: ১০, উত্তর মডেল টাউন, উতরা,ডাকঘর: উতর - ১২৩০, উতরা পশ্চম, ঢাকা"

result = corrector.correct(address)

print("Input:    ", result["input_address"])
print("Corrected:", result["corrected_address"])
display(highlight(result))
display(table([
    {"field": f["title"], "extracted": f["extracted"], "corrected": f["corrected"], "changed": f["changed"],
     "filter": f["filter"] or "", "searched": f"{f['searched']} / {f['total']}", "score": f["score"],
     "vector top": f["vector_top"], "re-rank overrode vector": f["rerank_changed_winner"]}
    for f in result["fields"]
]))
print("Timings (ms):", result["timings_ms"])

Input:     রহমান মঞ্জিল, বাড়ি নং: ৩৪, রোড নং: ৬, সেক্টর: ১০, উত্তর মডেল টাউন, উতরা,ডাকঘর: উতর - ১২৩০, উতরা পশ্চম, ঢাকা
Corrected: রহমান মঞ্জিল, বাড়ি নং: ৩৪, রোড নং: ৬, সেক্টর: ১০, উত্তর মডেল টাউন, উতরা,ডাকঘর: উত্তরা - ১২৩০, উত্তরা পশ্চিম, ঢাকা


field,extracted,corrected,changed,filter,searched,score,vector top,re-rank overrode vector
District,ঢাকা,ঢাকা,False,,68 / 68,1.0000,ঢাকা,False
Thana,উতরা পশ্চম,উত্তরা পশ্চিম,True,{'district': 'ঢাকা'},77 / 915,0.7890,উত্তরা পূর্ব,True
Post office,উতর,উত্তরা,True,{'district': 'ঢাকা'},303 / 10126,0.8388,আতি,True


Timings (ms): {'tokenize': 0.06, 'extract': 0.25, 'embed': 520.72, 'vector_search': 50.04, 'rerank': 5.52, 'total': 591.33}


### Candidates per field

Top vector search hits next to the re-ranked list. `vector_rank` shows where each re-ranked candidate sat in the vector results; re-ranking often promotes a candidate the embedding search ranked lower. Thana candidates all come from the corrected district.

In [13]:
TOP = 5
for field in result["fields"]:
    scope = f" · filter {field['filter']}" if field["filter"] else ""
    display(HTML(f"<h4>{escape(field['title'])}: <code>{escape(field['query_text'])}</code>"
                 f" · searched {field['searched']} of {field['total']}{escape(scope)}</h4>"))
    vector_rows = [{**c, "district": c["metadata"].get("district", "")} for c in field["vector_candidates"][:TOP]]
    display(table(vector_rows, ["rank", "name", "district", "distance", "similarity"]))
    display(table(field["reranked"][:TOP],
                  ["rank", "name", "score", "unigram", "bigram", "length", "distance", "vector_rank"]))

rank,name,district,distance,similarity
1,ঢাকা,,0.0000,1.0000
2,দিনাজপুর,,0.1888,0.8112
3,গাজীপুর,,0.2073,0.7927
4,চট্টগ্রাম,,0.2120,0.7880
5,নারায়ণগঞ্জ,,0.2168,0.7832


rank,name,score,unigram,bigram,length,distance,vector_rank
1,ঢাকা,1.0000,4,3,4,0.0000,1
2,ঠাকুরগাঁও,0.6842,3,1,9,0.2675,25
3,কক্সবাজার,0.6677,3,0,9,0.2299,11
4,নারায়ণগঞ্জ,0.6340,2,0,11,0.2168,5
5,রাজশাহী,0.6310,2,0,7,0.2210,9


rank,name,district,distance,similarity
1,উত্তরা পূর্ব,ঢাকা,0.2749,0.7251
2,মুগদা,ঢাকা,0.2814,0.7186
3,উত্তরা,ঢাকা,0.2901,0.7099
4,পল্টন,ঢাকা,0.2915,0.7085
5,ডেমরা,ঢাকা,0.2961,0.7039


rank,name,score,unigram,bigram,length,distance,vector_rank
1,উত্তরা পশ্চিম,0.7890,9,7,12,0.3015,7
2,উত্তরা পূর্ব,0.6951,6,4,11,0.2749,1
3,উত্তরা,0.6282,4,3,6,0.2901,3
4,উত্তরখান,0.6204,5,2,8,0.3013,6
5,সূত্রাপুর,0.6149,5,2,9,0.3090,12


rank,name,district,distance,similarity
1,আতি,ঢাকা,0.2289,0.7711
2,উত্তরা,ঢাকা,0.2303,0.7697
3,বেউটা,ঢাকা,0.2354,0.7646
4,আউনা,ঢাকা,0.2408,0.7592
5,আটি,ঢাকা,0.2475,0.7525


rank,name,score,unigram,bigram,length,distance,vector_rank
1,উত্তরা,0.8388,3,2,6,0.2303,2
2,উত্তরখান,0.8100,3,2,8,0.2714,15
3,উত্তর গাজীর হাট,0.8020,3,2,13,0.2829,20
4,উজামপুর,0.6393,2,0,7,0.2582,9
5,তুরাগ,0.6283,2,0,5,0.2739,16


### Filtered vs unfiltered thana search

The same thana query, once over all thana records and once over the corrected district only (what the pipeline does).

In [14]:
import time

thana_result = next(f for f in result["fields"] if f["collection"] == "thana")
query_embedding = corrector.embedder.encode([thana_result["query_text"]])[0]

rows = []
for where in [None, thana_result["filter"]]:
    start = time.perf_counter()
    hits = corrector.store.query(thana_field, query_embedding, settings.n_results, where=where)
    ms = (time.perf_counter() - start) * 1000
    rows.append({"filter": where or "none", "searched": corrector.store.count(thana_field, where),
                 "ms": round(ms, 2), "top 5": ", ".join(h["name"] for h in hits[:5])})
display(table(rows))

filter,searched,ms,top 5
none,915,5.0800,"পরশুরাম, পরশুরাম, পোরশা, পোরশা, কয়রা"
{'district': 'ঢাকা'},77,5.9600,"উত্তরা পূর্ব, মুগদা, উত্তরা, পল্টন, ডেমরা"


### Step log

In [15]:
display(table(result["logs"], ["elapsed_ms", "level", "step", "message"]))

elapsed_ms,level,step,message
0.0500,info,input,Received address (108 characters)
0.1500,info,tokenize,Split address into 67 Bangla character units (non-Bangla characters dropped) in 0.06 ms
0.3700,info,extract,[District] Found 'ঢাকা' by position (last remaining part of the address)
0.4000,info,extract,[Thana] Found 'উতরা পশ্চম' by position (last remaining part of the address)
0.4100,info,extract,[Post office] Found 'উতর' by label ('ডাকঘর:')
521.2500,info,embed,Encoded 3 queries with BAAI/bge-m3 in 520.72 ms
529.9000,info,vector_search,"[District] Searched 68 of 68 records, retrieved 30 candidates in 6.32 ms; nearest 'ঢাকা' (distance 0.0000)"
532.2300,info,rerank,[District] Re-ranked 30 candidates in 2.2 ms; best 'ঢাকা' (score 1.000 = 0.3 x sliding 1.000 + 0.7 x similarity 1.000)
532.2400,success,decision,[District] 'ঢাকা' is already correct
532.2900,info,filter,[Thana] Filtering by district = 'ঢাকা'


In [ ]:
# INFERENCE PIPELINE
my_address = "ডাকঘর: ধানমন্ডী, থানা: ধানমন্ডি, জেলা: ঢাকা"

my_result = corrector.correct(my_address)
display(highlight(my_result))
my_result["corrections"]

{'district': 'ঢাকা', 'thana': 'ধানমন্ডি', 'post_office': 'ধানমন্ডি'}